# Data Warehousing Concepts

## Overview
A data warehouse is an analytical database optimised for read-heavy queries across large volumes of historical data. It uses a different design philosophy from transactional (OLTP) databases: denormalised star schemas, wide fact tables, and pre-built calendar dimensions.

**OLTP vs OLAP:**

| Aspect | OLTP (transactional) | OLAP (analytical / DWH) |
|---|---|---|
| Primary use | Insert/update individual records | Aggregate millions of records |
| Schema | 3NF normalised (many tables) | Star/snowflake (few fat tables) |
| Query pattern | Point lookups, small result sets | Full scans, GROUP BY, window functions |
| Indexes | Many, on FK and lookup columns | Fewer; partition + BRIN/GIN |
| Row count | Thousands to millions | Hundreds of millions to billions |
| Update pattern | Frequent INSERT/UPDATE/DELETE | Append-only facts; SCD dimensions |

**This notebook's dataset:** an insurance/finance DWH with `fact_policy`, `dim_date`, `dim_customer`, `dim_product`, and `dim_agent` — fully runnable in SQLite with PostgreSQL patterns shown throughout.

---

In [ ]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


---
## Star schema vs snowflake schema

In [ ]:
print("=== Star schema vs Snowflake schema ===")
print()

print("Star schema (this notebook's design):")
print("""
                    dim_date
                       │
   dim_agent ──── fact_policy ──── dim_customer
                       │
                   dim_product

  - Fact table at centre: fact_policy
  - Dimension tables radiate outward (one level)
  - Denormalised dimensions (e.g., product_line IN dim_product, not a separate table)
  - Fewer JOINs → faster analytical queries
  - Slightly more storage (repeated dimension values)
""")

print("Snowflake schema (normalised dimensions):")
print("""
   dim_region ── dim_customer ──── fact_policy ──── dim_product ── dim_product_line
                                        │
                                    dim_date

  - Dimensions are normalised into sub-dimensions
  - e.g., dim_product → dim_product_line (separate table for product lines)
  - Less storage, more JOINs
  - Harder to query; BI tools handle it better than raw SQL
""")

comparison = [
    ("Joins",          "Few (1 per dim)",    "Many (2-5+ per dim)"),
    ("Query speed",    "Faster",             "Slower (more joins)"),
    ("Storage",        "More (denorm)",      "Less (normalised)"),
    ("Maintainability","Simpler",            "More complex"),
    ("Typical use",    "Most DWH (default)", "Very large dims, strict 3NF teams"),
]
print(f"  {'Aspect':<18s}  {'Star':<22s}  Snowflake")
print("  " + "-"*55)
for aspect, star, snow in comparison:
    print(f"  {aspect:<18s}  {star:<22s}  {snow}")


---
## Facts vs dimensions

In [ ]:
print("=== Facts vs Dimensions ===")
print()

print("Fact tables:")
fact_props = [
    ("Content",       "Numeric measures (premium, claim, count)"),
    ("Grain",         "One row per business event at lowest level of detail"),
    ("Foreign keys",  "Surrogate keys into every dimension table"),
    ("Size",          "Very large — millions to billions of rows"),
    ("Updates",       "Append-only (new events added; old rows not changed)"),
    ("Additive",      "Measures can be SUMmed across any dimension"),
    ("Semi-additive", "Some measures valid for some dims (e.g., balance by date only)"),
    ("Non-additive",  "Ratios, percentages — cannot be SUMmed"),
]
for prop, desc in fact_props:
    print(f"  {prop:<14s}  {desc}")

print()
print("Dimension tables:")
dim_props = [
    ("Content",       "Descriptive attributes (names, categories, dates)"),
    ("Grain",         "One row per entity (one per customer, one per product)"),
    ("Primary key",   "Surrogate integer key (not the source system ID)"),
    ("Size",          "Smaller — thousands to millions of rows"),
    ("Updates",       "SCD handling required (history or overwrite)"),
    ("Hierarchy",     "Multiple levels baked in (year > quarter > month > day in dim_date)"),
]
for prop, desc in dim_props:
    print(f"  {prop:<14s}  {desc}")

print()
print("Our insurance DWH grain:")
grain = [
    ("fact_policy",   "One row per insurance policy issued (policy-grain)"),
    ("dim_date",      "One row per calendar day (2022–2024)"),
    ("dim_customer",  "One row per customer (current version; SCD2 for history)"),
    ("dim_product",   "One row per insurance product"),
    ("dim_agent",     "One row per sales agent"),
]
for table, desc in grain:
    print(f"  {table:<16s}  {desc}")


---
## Surrogate keys

In [ ]:
print("=== Surrogate keys vs natural keys ===")
print()

print("Why surrogate keys?")
surrogate_reasons = [
    ("Source system independence",
     "Customer IDs change when companies merge; surrogate keys never change"),
    ("SCD history",
     "SCD Type 2 gives one customer multiple rows; surrogate key differentiates versions"),
    ("Join performance",
     "Integer surrogate keys join faster than VARCHAR natural keys"),
    ("NULL handling",
     "Unknown/N/A dimension gets surrogate key -1 (not NULL) — avoids NULL FK issues"),
    ("Multi-source integration",
     "Two source systems may have same customer_id; surrogate key is always unique"),
]
for reason, desc in surrogate_reasons:
    print(f"  {reason:<30s}  {desc}")

print()
# Show our surrogate vs natural key setup
rows = conn.execute("""
    SELECT c.customer_key, c.customer_id, c.full_name, c.province, c.risk_tier
    FROM   dim_customer c
    ORDER  BY c.customer_key
""").fetchall()
print("dim_customer — surrogate vs natural key:")
print(f"  {'customer_key':<14s}  {'customer_id':<12s}  {'full_name':<20s}  province  risk_tier")
print("  " + "-"*65)
for r in rows:
    print(f"  {r['customer_key']:<14d}  {r['customer_id']:<12s}  {r['full_name']:<20s}  {r['province']:<9s}  {r['risk_tier']}")

print()
print("The -1 / 'Unknown' surrogate pattern:")
print("""
  -- Insert unknown dimension member before loading facts:
  INSERT INTO dim_customer (customer_key, customer_id, full_name, segment, risk_tier)
  VALUES (-1, 'UNKNOWN', 'Unknown Customer', 'unknown', 'unknown');

  -- Facts with no matching customer get FK = -1 (not NULL):
  INSERT INTO fact_policy (..., customer_key, ...)
  SELECT ...,
         COALESCE(c.customer_key, -1),   -- -1 if no match
         ...
  FROM   stg_policy_load s
  LEFT JOIN dim_customer c ON s.customer_id = c.customer_id;
""")


---
## Common Pitfalls

**1. Choosing the wrong grain for the fact table**
The grain defines one row = one what. `fact_policy` at policy-grain means one row per policy issued. If claims are on separate rows, a different fact table (`fact_claim`) is needed. Mixing grains in one fact table (some rows = policies, some = claims) causes double-counting and incorrect aggregations.

**2. Using natural keys as foreign keys in fact tables**
Storing `customer_id = 'C001'` in the fact table instead of the surrogate `customer_key = 1` breaks SCD Type 2 history. When customer C001 gets a new surrogate key (after a risk tier change), historical facts still point to the old surrogate — which is the correct SCD2 behaviour. Natural keys cannot do this.

**3. Putting too many measures in one fact table**
Combining policy premiums and claim events in the same fact table — when claims happen infrequently — creates mostly-NULL rows for claim columns on policy rows. Separate fact tables (`fact_policy`, `fact_claim`) at the appropriate grain are cleaner.

**4. Storing calculated ratios in the fact table**
Storing `loss_ratio_pct` as a column is an anti-pattern. Loss ratio is calculated from premiums and claims. Storing it pre-calculated means it goes stale when source data is corrected. Compute derived measures in queries or materialized views.

**5. Building the dim_date table at query time**
Joining `EXTRACT(YEAR FROM policy_date) = 2024` in every query is slower and less flexible than a pre-built `dim_date` with year, quarter, month, is_weekend, is_holiday columns. Always build dim_date once, populated for 10+ years ahead.

**6. Snowflaking a small dimension**
Splitting `dim_product` into `dim_product` + `dim_product_line` adds a JOIN for every product query but saves only a few KB of storage (product lines are small). Only snowflake dimensions that are genuinely large (millions of rows) or change independently at different rates.


---
*sql_methods_library - Samantha McGarrigle*